## Data

This project uses Earnings-21, a set of 44 English earnings calls with human reference transcripts, loaded from the Hugging Face ASR leaderboard long-form collection. Earnings calls are a realistic captioning use case since they mix multiple speakers, numbers, company names and financial terms.

The analysis uses the first two of the ten data files, which hold 10 calls and about 10.75 hours of audio across six sectors. TED-LIUM was the original choice, but it is no longer available through Hugging Face.

In [2]:
from huggingface_hub import hf_hub_download
from datasets import load_dataset

# load the two earnings-21 files that already finished downloading
repo = "hf-audio/asr-leaderboard-longform"
names = ["earnings21/test-00000-of-00010.parquet", "earnings21/test-00001-of-00010.parquet"]
paths = [hf_hub_download(repo, n, repo_type="dataset") for n in names]

ds = load_dataset("parquet", data_files=paths, split="train")

print(ds)
print()

for s in ds.remove_columns("audio"):
    print(s["company_name"], s["sector"], s["unique_speakers"], round(s["audio_length"] / 60), "min")

Dataset({
    features: ['audio', 'audio_length', 'original_sample_rate', 'company_name', 'financial_quarter', 'sector', 'speaker_switches', 'unique_speakers', 'curator_id', 'text'],
    num_rows: 10
})

Monro Inc Consumer Goods 10 55 min
Culp Inc Industrial Goods 8 41 min
General Electric Conglomerate 14 96 min
Danaher Corp Conglomerate 7 45 min
Spire Inc Utilities 10 55 min
Ingersoll Rand Industrial Goods 14 66 min
Cementos Argos Industrial Goods 20 78 min
Kuehne Nagel International Services 13 81 min
Constellium Industrial Goods 10 63 min
Travelers Companies Inc Financial 15 65 min


## Transcription test

Before transcribing all 10 calls, this cell runs Whisper on the first two minutes of the Culp Inc call, which is the shortest in the subset. It checks how long transcription takes on a Mac and gives a first look at how Whisper’s output compares with the human reference.

In [3]:
import time
import whisper
from datasets import Audio

# get the culp call as raw audio bytes so whisper can decode it with ffmpeg
ds_raw = ds.cast_column("audio", Audio(decode=False))
idx = list(ds["company_name"]).index("Culp Inc")
culp = ds_raw[idx]

with open("culp.wav", "wb") as f:
    f.write(culp["audio"]["bytes"])

# load the audio at 16 kHz and keep the first two minutes
audio = whisper.load_audio("culp.wav")
clip = audio[: 16000 * 120]

model = whisper.load_model("small.en")

start = time.time()
result = model.transcribe(clip, fp16=False)
elapsed = time.time() - start

print(round(elapsed), "seconds to transcribe 2 minutes of audio")
print(round(elapsed / 2 * 645 / 60, 1), "estimated hours for all 10 calls")
print()
print(result["text"])
print()
print(culp["text"][:1500])

100%|███████████████████████████████████████| 461M/461M [01:03<00:00, 7.63MiB/s]


15 seconds to transcribe 2 minutes of audio
81.8 estimated hours for all 10 calls

 Good day and welcome to Culp's third quarter 2020 earnings conference call. Today's call is being recorded at this time for opening remarks and introductions. I'd like to turn the call over to Ms. Drew Anderson. Please go ahead. Thank you. Good morning and welcome to the Culp conference call to review the company's results for the third quarter of fiscal 2020. As we start, let me state that this morning's call will contain forward-looking statements about the business, financial condition, and prospects of the company. Forward-looking statements are statements that include projections, expectations, or beliefs about future events or results, or otherwise are not statements of historical fact. The actual performance of the company could differ materially from that indicated by the forward-looking statements because of various risks and uncertainties. These risks and uncertainties are described in our reg

## Transcribing all calls

The test showed that the small.en model transcribes about 8 times faster than real time, so all 10 calls take roughly 78 minutes. That makes small.en a good balance of speed and accuracy for this project.

Each finished call is saved to whisper_transcripts.csv along with its reference transcript and metadata. If the run is interrupted, rerunning the cell skips calls that are already saved.

In [5]:
import os
import pandas as pd

# load any transcripts saved from an earlier run so finished calls are skipped
rows = []
if os.path.exists("whisper_transcripts.csv"):
    rows = pd.read_csv("whisper_transcripts.csv").to_dict("records")
done = {r["company_name"] for r in rows}

for i in range(len(ds_raw)):
    call = ds_raw[i]
    name = call["company_name"]
    if name in done:
        continue

    with open("call.wav", "wb") as f:
        f.write(call["audio"]["bytes"])
    audio = whisper.load_audio("call.wav")

    start = time.time()
    result = model.transcribe(audio, fp16=False)
    minutes = round((time.time() - start) / 60, 1)

    rows.append({
        "company_name": name,
        "sector": call["sector"],
        "unique_speakers": call["unique_speakers"],
        "speaker_switches": call["speaker_switches"],
        "audio_minutes": round(call["audio_length"] / 60, 1),
        "transcribe_minutes": minutes,
        "whisper_text": result["text"],
        "reference_text": call["text"],
    })

    # save after every call so progress is never lost
    pd.DataFrame(rows).to_csv("whisper_transcripts.csv", index=False)
    print(name, "done in", minutes, "minutes")

Monro Inc done in 7.0 minutes
Culp Inc done in 5.6 minutes
General Electric done in 11.8 minutes
Danaher Corp done in 5.5 minutes
Spire Inc done in 7.0 minutes
Ingersoll Rand done in 8.6 minutes
Cementos Argos done in 8.1 minutes
Kuehne Nagel International done in 9.0 minutes
Constellium done in 7.7 minutes
Travelers Companies Inc done in 7.9 minutes
